# Before you launch the job

Five checks.  Each one takes seconds and each one catches a failure that otherwise
shows up an hour into a run, or — worse — not at all.

**This page needs a GPU.**  `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.  Q1 and Q4 measure the card, so without one they have nothing to
report.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/infer.py', 'lib/preflight.py', 'lib/srcheck.py',
          'lib/sr_models.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'infer', 'preflight', 'srcheck', 'sr_models'):
    sys.modules.pop(m, None)

import numpy as np
from infer import *
from preflight import *
from srcheck import structure, geometry, radiometry
from sr_models import load_edsr

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/condition/clear.tif', 'scene.tif')
net = load_edsr(get('models/14_inference/checkpoints/edsr_x2.pt', 'edsr_x2.pt'),
                scale=2)
cfg = Config(scale=2, tile=256, overlap=16, batch=4, precision='fp32')

## Q1.  What GPU is this, and will it run out of memory?

`Runtime > Change runtime type` decides which card you get, and the card decides
everything below.

In [ ]:
gpu_state(cfg.device)

Whether a config fits is not a guess.  Memory grows in a straight line with the
pixels in one forward pass, so two cheap runs measure the slope and the rest is
arithmetic.

In [ ]:
lr, window = to_8bit(PATH)
per, cap = budget(lr, net, cfg)

In [ ]:
show_budget(lr, net, cfg)

The table says the largest batch each tile size can take on **this** card.  Anything
under the green line runs; anything over it is the `CUDA out of memory` you would have
met at tile 400 of 900.

## Q2.  Is the data in the form the model expects?

The scene is 12-bit in a `uint16` container.  The model was trained on 0 - 255.  Those
are not the same numbers, and nothing in the code will say so.

In [ ]:
lr, window = to_8bit(PATH)

for i, (a, b) in enumerate(zip(*window), start=1):
    print(f'band {i}   2 - 99.5 % window   {a:.0f} - {b:.0f}')
print('input  ', lr.shape, lr.dtype, lr.min(), '-', lr.max())

In [ ]:
show_conversion(PATH, net, cfg)

Panel 3 is the same model on the same scene with the conversion skipped.  No error, no
warning — 60 % of the output lands above 255 and clips to white.

Keep the window.  Without `lo` and `hi` the result cannot be turned back into the
sensor's units later.

## Q3.  Patch size and overlap

Tile size is bounded above by Q1 and below by the overhead of running many small
patches.  Overlap is the margin read and thrown away so that edge pixels had
neighbours.

In [ ]:
show_tiles(lr.shape, (128, 256, 512), cfg)

In [ ]:
show_overlap(lr.shape, (0, 16, 64), cfg)

Bigger tiles waste less margin — `+33 %` of the pixels at tile 128 against `+11 %` at
512 — and the margin itself has to be large enough that the network's receptive field
is covered.  For this EDSR, 16 is enough and 32 changes nothing.

## Q4.  Precision and batch size

These are the two knobs that move memory, and Q1 already said how much room there is.

In [ ]:
show_batch(lr.shape, (1, 4, 16), cfg)

In [ ]:
for p in ('fp32', 'fp16'):
    try_run(lr, net, replace(cfg, precision=p, batch=8))

`fp16` halves the memory and runs faster, and on this model the output differs by at
most one level.  It is the cheapest thing on this page.

## Q5.  The script

Everything above collapses into five lines.  Convert, run, convert back, write with the
input's georeferencing.

In [ ]:
cfg = replace(cfg, precision='fp16')

lr, (lo, hi) = to_8bit(PATH)
sr = sr_tiled(lr, net, cfg, report=True)
sr12 = np.clip(sr / 255 * (hi - lo) + lo, 0, 65535).round().astype('uint16')
OUT = save_sr(PATH, sr12, 'sr.tif', scale=cfg.scale)

Written is not the same as correct.  The checks from the output-review page answer
that in three passes.

In [ ]:
structure(PATH, OUT)
print()
geometry(PATH, OUT)

In [ ]:
radiometry(PATH, OUT)

Structure and geometry are clean: twice the pixels, same CRS, pixel size halved, origin
identical to the last decimal.

Radiometry shows two things worth noticing.  The maximum dropped from 4095 to 972 —
that is the 2 - 99.5 % window, not the model, and it is the price of Q2.  And the
standard deviation fell 2 - 4 %, where a sharpening model should raise it.  This
demonstration model was trained on a different sensor, so on THEOS data it smooths
slightly.  The check caught it; that is what the check is for.